In [1]:
using SpecialFunctions
using LinearAlgebra
using Plots, LaTeXStrings
using Struve
using KrylovKit
using SparseArrays
using ForwardDiff
using FFTW
using Statistics
using Measures
using Base.Threads
using TensorOperations
using CSV, DataFrames
include("module_mos2_plot3.jl")
using .exciton

In [2]:
Nsample=60
sz = 1
lattice = TMDLattice();
VInt = InteractionMatrix(lattice, Nsample; lambda = 0.667, r0 = 4.288)
bi_1 = lattice.b1 / Nsample
bi_2 = lattice.b2 / Nsample
bi_3 = -bi_1 - bi_2
wb = 2 / (3 * norm(bi_1)^2)

strainlist = -6:0.5:6

wannierlist = Vector{ComplexF64}(undef, length(strainlist))
blochlist = Vector{Vector{ComplexF64}}(undef, length(strainlist))

Threads.@threads for i in 1:length(strainlist)
	strain_val = strainlist[i]

	epsilonyy = -(sqrt((1 + strain_val / 100)^2 * 3 / 4 + 1 / 4) - 1) * 5
	println("Thread $(Threads.threadid()) doing strain $strain_val (Index $i)")
	flush(stdout)
	local_sys = TMDBSE(lattice, [0.0, 0], Nsample; sz, epsilonyy)
	local_sys = add_bse_kernel(local_sys, VInt)

	bsemat = local_sys.BSEKernel
	xlen = size(bsemat)[1]
	x0 = rand(xlen) 

	valslist, vecslist, info = eigsolve(bsemat, x0, 10, :SR,
		krylovdim = 40, tol = 1e-10, maxiter = 20,
		verbosity = 0, ishermitian = true)

	local_sys = add_wannier(local_sys)
	psik = vecslist[1]
	psi_real = exciton.envelope_real_fft(psik, local_sys)
	r1 = Rshift_calculator(local_sys, psi_real, div(Nsample, 2) - 2)

	wannierlist[i] = r1[2]
	flush(stdout)

	wv, wc1, wc2 = local_sys.Wannier.valence, local_sys.Wannier.conduction1, local_sys.Wannier.conduction2

	blochlist[i] = exciton.exciton_subroutine_4th(wv, wc1, wc2; state = 1, polarization = [[cos(2pi / 3), sin(2pi / 3)], [cos(3pi / 4), sin(3pi / 4)], [cos(5pi / 6), sin(5pi / 6)]], VInt, lattice, epsilonyy, sz = 1, Nsample)
end

polarization_list=Vector{Float64}(undef, length(strainlist))
for i ∈ 1:length(strainlist)
	polarization_list[i]=real(blochlist[i][1])
end

polarization_list2=Vector{Float64}(undef, length(strainlist))
for i ∈ 1:length(strainlist)
	polarization_list2[i]=real(blochlist[i][2])
end

polarization_list3=Vector{Float64}(undef, length(strainlist))
for i ∈ 1:length(strainlist)
	polarization_list3[i]=real(blochlist[i][3])
end

sigma1=polarization_list .- real.(wannierlist)
println("std1 is ", sqrt(sum(sigma1 .^ 2 ./ 25)))
sigma2=polarization_list2 .- real.(wannierlist)
println("std2 is ", sqrt(sum(sigma2 .^ 2 ./ 25)))
sigma3=polarization_list3 .- real.(wannierlist)
println("std3 is ", sqrt(sum(sigma3 .^ 2 ./ 25)))

df = DataFrame(
    strainlist = strainlist,
    rshift_wannier = real.(wannierlist),
    rshift_full_2pi_3 = polarization_list,
    rshift_full_3pi_4 = polarization_list2,
    rshift_full_5pi_6 = polarization_list3
)

CSV.write("figs2.csv", df)

Thread 3 doing strain -6.0 (Index 1)
Thread 5 doing strain 3.5 (Index 20)
Thread 2 doing strain -2.5 (Index 8)
Thread 4 doing strain 0.5 (Index 14)
gauge fixing successful!
gauge fixing successful!
gauge fixing successful!
gauge fixing successful!
[1.631120930237e-7, 0.0002852699908534634]
[1.1362470106694231e-7, -0.0013644937695126318]
[3.015163183001174e-7, 0.0019326979106397825]
[1.0841132173501994e-7, -0.0028666240152033736]
rtot for all polarizations: ComplexF64[0.00028485044133501954 + 2.0510897944977204e-8im, 0.0002848565825911053 + 2.0510897944977204e-8im, 0.00028486275017505775 + 2.0510897944977204e-8im]
Thread 4 doing strain 1.0 (Index 15)
gauge fixing successful!
rtot for all polarizations: ComplexF64[-0.001365042627230887 + 2.0565378262408568e-8im, -0.0013650337105539888 + 2.0565378262408568e-8im, -0.0013650244929084875 + 2.0565378262408568e-8im]
Thread 2 doing strain -2.0 (Index 9)
gauge fixing successful!
rtot for all polarizations: ComplexF64[-0.0028674201968834565 + 2.1

"figs2.csv"